# Simple AI Question-Answer Chatbot for Google Colab

This notebook creates a simple chatbot that can answer many general questions using:

- Wikipedia summaries for factual questions
- DuckDuckGo Instant Answer for quick facts
- Safe math evaluation for arithmetic questions

It uses only Python standard-library modules, so it should run in Colab without installing extra packages.

> Note: No chatbot can guarantee perfect accuracy for every question. This bot gives concise answers and includes a source link when available.

In [ ]:
import ast
import json
import math
import operator
import re
import textwrap
import urllib.parse
import urllib.request

print('✅ Chatbot dependencies loaded successfully.')

## Chatbot Code

In [ ]:
USER_AGENT = 'SimpleColabChatbot/1.0 (educational notebook)'

ALLOWED_OPERATORS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.FloorDiv: operator.floordiv,
    ast.Mod: operator.mod,
    ast.Pow: operator.pow,
    ast.USub: operator.neg,
    ast.UAdd: operator.pos,
}

SMALL_TALK = {
    'hello': 'Hello! Ask me any simple factual question.',
    'hi': 'Hi! Ask me a question and I will try to answer accurately.',
    'hey': 'Hey! What would you like to know?',
    'who are you': 'I am a simple Python chatbot that answers questions using public web knowledge sources.',
    'what can you do': 'I can answer many factual questions, solve simple math, and provide source links when possible.',
    'help': 'Ask a question like: What is photosynthesis? Who was Ada Lovelace? What is 25 * 18?',
}

def fetch_json(url, timeout=12):
    request = urllib.request.Request(url, headers={'User-Agent': USER_AGENT})
    with urllib.request.urlopen(request, timeout=timeout) as response:
        return json.loads(response.read().decode('utf-8'))

def safe_eval_math(expression):
    expression = expression.replace('^', '**')
    parsed = ast.parse(expression, mode='eval')

    def evaluate(node):
        if isinstance(node, ast.Expression):
            return evaluate(node.body)
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in ALLOWED_OPERATORS:
            return ALLOWED_OPERATORS[type(node.op)](evaluate(node.left), evaluate(node.right))
        if isinstance(node, ast.UnaryOp) and type(node.op) in ALLOWED_OPERATORS:
            return ALLOWED_OPERATORS[type(node.op)](evaluate(node.operand))
        raise ValueError('Only basic arithmetic is allowed.')

    result = evaluate(parsed)
    if isinstance(result, float) and result.is_integer():
        return int(result)
    return result

def looks_like_math(question):
    cleaned = question.lower().strip()
    cleaned = cleaned.replace('what is', '').replace('calculate', '').replace('solve', '')
    cleaned = cleaned.replace('plus', '+').replace('minus', '-')
    cleaned = cleaned.replace('times', '*').replace('multiplied by', '*')
    cleaned = cleaned.replace('divided by', '/')
    cleaned = cleaned.strip(' ?')
    return bool(re.fullmatch(r'[0-9+\-*/(). %^]+', cleaned)), cleaned

def duckduckgo_answer(question):
    query = urllib.parse.urlencode({
        'q': question,
        'format': 'json',
        'no_html': 1,
        'skip_disambig': 1,
    })
    data = fetch_json(f'https://api.duckduckgo.com/?{query}')
    answer = data.get('Answer') or data.get('AbstractText')
    source = data.get('AbstractURL') or data.get('AnswerType')
    if answer:
        return answer, source
    return None, None

def wikipedia_search(question):
    search_url = 'https://en.wikipedia.org/w/api.php?' + urllib.parse.urlencode({
        'action': 'query',
        'list': 'search',
        'srsearch': question,
        'format': 'json',
        'srlimit': 1,
    })
    search_data = fetch_json(search_url)
    results = search_data.get('query', {}).get('search', [])
    if not results:
        return None, None

    title = results[0]['title']
    summary_url = 'https://en.wikipedia.org/api/rest_v1/page/summary/' + urllib.parse.quote(title)
    summary_data = fetch_json(summary_url)
    summary = summary_data.get('extract')
    page_url = summary_data.get('content_urls', {}).get('desktop', {}).get('page')
    return summary, page_url

def short_answer(text, max_sentences=3):
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    return ' '.join(sentences[:max_sentences])

def chatbot(question):
    question = question.strip()
    lowered = question.lower()

    if not question:
        return 'Please type a question.'

    if lowered in {'bye', 'exit', 'quit'}:
        return 'Goodbye! 👋'

    for phrase, response in SMALL_TALK.items():
        if phrase == lowered:
            return response

    is_math, math_expression = looks_like_math(question)
    if is_math:
        try:
            return f'The answer is {safe_eval_math(math_expression)}.'
        except Exception as error:
            return f'I could not solve that math expression: {error}'

    try:
        answer, source = duckduckgo_answer(question)
        if answer:
            reply = short_answer(answer)
            if source and str(source).startswith('http'):
                reply += f'\n\nSource: {source}'
            return reply

        answer, source = wikipedia_search(question)
        if answer:
            reply = short_answer(answer)
            if source:
                reply += f'\n\nSource: {source}'
            return reply

        return 'I could not find a reliable simple answer. Try asking with more specific words.'
    except Exception as error:
        return 'I could not access online sources right now. Please check the internet connection and try again. Details: ' + str(error)

print('✅ Chatbot is ready!')

## Ask One Question

In [ ]:
question = input('Ask me anything: ')
print('
Bot:')
print(textwrap.fill(chatbot(question), width=100))

## Continuous Chat Mode
Run this cell to keep chatting. Type `bye`, `exit`, or `quit` to stop.

In [ ]:
print('🤖 Simple AI Chatbot is running. Type bye, exit, or quit to stop.')

while True:
    user_question = input('You: ')
    bot_answer = chatbot(user_question)
    print('Bot:', textwrap.fill(bot_answer, width=100))
    if user_question.strip().lower() in {'bye', 'exit', 'quit'}:
        break